##**Step 1**: Install Spark

###**Step 2**: Setting up Spark

Before you can connect to a Spark cluster, Spark needs to be installed. The code below is boilerplate code that can be used to set-up Spark. Please note that this code will be leveraged in all the notebooks since each nodebook is a separate entity.

### **Step 3**. Import the lib




In [1]:
# Colab-friendly setup for PySpark
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q "pyspark[connect]==3.5.1" "dataproc-spark-connect==0.8.3"

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("RDDPractice").getOrCreate()
sc = spark.sparkContext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.2 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Step 4**. Import the Data

In [6]:
df = spark.read.option("header", "true").csv("/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Dataset/books.csv")

In [ ]:
df.show()

+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+
|bookID|               title|             authors|average_rating|      isbn|       isbn13|language_code|  num_pages|ratings_count|text_reviews_count|publication_date|           publisher|
+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+
|     1|Harry Potter and ...|J.K. Rowling/Mary...|          4.57|0439785960|9780439785969|          eng|        652|      2095690|             27591|       9/16/2006|     Scholastic Inc.|
|     2|Harry Potter and ...|J.K. Rowling/Mary...|          4.49|0439358078|9780439358071|          eng|        870|      2153167|             29221|        9/1/2004|     Scholastic Inc.|
|     4|Harry Potter and ...|        J.K. Rowling|          

In [7]:
#Display the data types of the columns

df.dtypes

[('bookID', 'string'),
 ('title', 'string'),
 ('authors', 'string'),
 ('average_rating', 'string'),
 ('isbn', 'string'),
 ('isbn13', 'string'),
 ('language_code', 'string'),
 ('  num_pages', 'string'),
 ('ratings_count', 'string'),
 ('text_reviews_count', 'string'),
 ('publication_date', 'string'),
 ('publisher', 'string')]

# Using SQL to interface with Spark

##**Step 1**: Register the DataFrame as a SQL temporary view so that you can interact with it using SQL commands


In [8]:
df.createOrReplaceTempView("books")

##**Step 2**: Query the temporary view using SQL

In [9]:
sqlDF = spark.sql("SELECT * FROM books")
sqlDF.show()

+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+
|bookID|               title|             authors|average_rating|      isbn|       isbn13|language_code|  num_pages|ratings_count|text_reviews_count|publication_date|           publisher|
+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+
|     1|Harry Potter and ...|J.K. Rowling/Mary...|          4.57|0439785960|9780439785969|          eng|        652|      2095690|             27591|       9/16/2006|     Scholastic Inc.|
|     2|Harry Potter and ...|J.K. Rowling/Mary...|          4.49|0439358078|9780439358071|          eng|        870|      2153167|             29221|        9/1/2004|     Scholastic Inc.|
|     4|Harry Potter and ...|        J.K. Rowling|          

##**Step 3**: Filter the data so that only records where the average rating is greater than 4.5

In [10]:
sqlDF.filter("average_rating > 4.5").show()

+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+
|bookID|               title|             authors|average_rating|      isbn|       isbn13|language_code|  num_pages|ratings_count|text_reviews_count|publication_date|           publisher|
+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+
|     1|Harry Potter and ...|J.K. Rowling/Mary...|          4.57|0439785960|9780439785969|          eng|        652|      2095690|             27591|       9/16/2006|     Scholastic Inc.|
|     5|Harry Potter and ...|J.K. Rowling/Mary...|          4.56|043965548X|9780439655484|          eng|        435|      2339585|             36325|        5/1/2004|     Scholastic Inc.|
|     8|Harry Potter Boxe...|J.K. Rowling/Mary...|          

##**Step 4**: Filter the dataset to include books that have an average rating of greater than 4.5 and the publisher is 'Ballantine Books'

In [11]:
sqlDF.filter("average_rating > 4.5").filter("publisher == 'Ballantine Books'").show()

+------+--------------------+--------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+----------------+
|bookID|               title|       authors|average_rating|      isbn|       isbn13|language_code|  num_pages|ratings_count|text_reviews_count|publication_date|       publisher|
+------+--------------------+--------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+----------------+
|    30|J.R.R. Tolkien 4-...|J.R.R. Tolkien|          4.59|0345538374|9780345538376|          eng|       1728|       101233|              1550|       9/25/2012|Ballantine Books|
+------+--------------------+--------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+----------------+



##**Step 5**: Add a new column that depicts for each book what proportion of the overall ratings count are text review counts

In [12]:
import pyspark.sql.functions as F

sqlDF = sqlDF.withColumn("proportion_of_ratings", F.bround((sqlDF.text_reviews_count/sqlDF.ratings_count),2))
sqlDF = sqlDF.withColumn("avg_rating_decimal", sqlDF.average_rating.cast('Decimal(4,2)'))
sqlDF.show()

+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+---------------------+------------------+
|bookID|               title|             authors|average_rating|      isbn|       isbn13|language_code|  num_pages|ratings_count|text_reviews_count|publication_date|           publisher|proportion_of_ratings|avg_rating_decimal|
+------+--------------------+--------------------+--------------+----------+-------------+-------------+-----------+-------------+------------------+----------------+--------------------+---------------------+------------------+
|     1|Harry Potter and ...|J.K. Rowling/Mary...|          4.57|0439785960|9780439785969|          eng|        652|      2095690|             27591|       9/16/2006|     Scholastic Inc.|                 0.01|              4.57|
|     2|Harry Potter and ...|J.K. Rowling/Mary...|          4.49|0439358078|97804393

# Aggregating data using GroupBy

##**Step 6**: Find the lowest average_rating where the publisher if 'Broadway Books'

In [13]:
sqlDF.filter("publisher == 'Broadway Books'").groupBy().min("avg_rating_decimal").show()


+-----------------------+
|min(avg_rating_decimal)|
+-----------------------+
|                   3.25|
+-----------------------+



##**Step 7**: Find the highest average_rating where the publisher if 'Broadway Books'


In [14]:
sqlDF.filter("publisher == 'Broadway Books'").groupBy().max("avg_rating_decimal").show()

+-----------------------+
|max(avg_rating_decimal)|
+-----------------------+
|                   4.43|
+-----------------------+



##**Step 8**: Ascertain the cumulative count of ratings received for Broadway Books

In [15]:
#First convert from string type to integer
sqlDF = sqlDF.withColumn("total_rating_count", sqlDF.ratings_count.cast('Integer'))
sqlDF.filter("publisher == 'Broadway Books'").groupBy().sum("total_rating_count").show()

+-----------------------+
|sum(total_rating_count)|
+-----------------------+
|                 695164|
+-----------------------+



##**Step 9:** Ascertain the average  count of ratings received for Broadway Books

In [16]:
sqlDF.filter("publisher == 'Broadway Books'").groupBy().avg("total_rating_count").show()

+-----------------------+
|avg(total_rating_count)|
+-----------------------+
|     14187.020408163266|
+-----------------------+



##**Step 10**: Show average count of ratings for each publisher

In [17]:
sqlDF.groupBy("publisher").avg("total_rating_count").show()


+--------------------+-----------------------+
|           publisher|avg(total_rating_count)|
+--------------------+-----------------------+
|       The New Press|                 1693.0|
|        Chosen Books|                  536.5|
|       Digireads.com|     24628.166666666668|
|               Ember|               351258.4|
|           IVP Books|                   10.5|
|       No Exit Press|                43253.8|
|Dabel Brothers Pr...|                  276.0|
|                 DAW|     5079.1463414634145|
|      Celestial Arts|                  442.0|
|Arcadia Publishin...|                    5.0|
|Hachette Littérature|                    1.0|
|         Cleis Press|                  428.5|
|Chicago Review Press|      6315.857142857143|
|                 HQN|                 7769.0|
|World Wrestling E...|                 1154.0|
|Wayne State Unive...|                   45.0|
|           Doubleday|      8012.916666666667|
|               Verso|     1204.8666666666666|
|   Time Life

We will use the **.agg()** method, which allows you to apply aggregate functions to columns in a DataFrame. This method enables you to pass an expression that uses various aggregation functions available in the `pyspark.sql.functions` submodule.

The `pyspark.sql.functions` submodule offers a wide range of useful functions to perform different types of computations, such as calculating standard deviations, sums, and averages. It's important to note that all the aggregation functions in this submodule require the name of a column from a `GroupedData` table as input, as shown in the example below.

##**Step 11**: Compute the standard deviation of ratings count by publisher

In [18]:
# Import pyspark.sql.functions as F
import pyspark.sql.functions as F

# Standard deviation of ratings count by publisher
sqlDF.groupBy("publisher").agg(F.stddev("ratings_count")).show()

+--------------------+---------------------+
|           publisher|stddev(ratings_count)|
+--------------------+---------------------+
|       The New Press|   4021.5151825694215|
|        Chosen Books|   457.49808742769625|
|       Digireads.com|   51201.525162505335|
|               Ember|    691000.7816122931|
|           IVP Books|   14.849242404917497|
|       No Exit Press|    17332.11097068098|
|Dabel Brothers Pr...|                 NULL|
|                 DAW|    6372.894532945668|
|      Celestial Arts|    612.3544725075502|
|Arcadia Publishin...|   2.9439202887759492|
|Hachette Littérature|                 NULL|
|         Cleis Press|   488.69792987761537|
|Chicago Review Press|    9251.830511283904|
|                 HQN|    7846.056844045931|
|World Wrestling E...|                 NULL|
|Wayne State Unive...|    46.66904755831214|
|           Doubleday|   24104.695025350382|
|               Verso|    1821.766994143977|
|   Time Life Medical|    40.95200442794793|
|         

# Joining datasets using Spark SQL

In [19]:
# Create the tables that will be joined

valuesA = [('Dog',1),('Monkey',2),('Elephant',3),('Penguin',4)]
TableA = spark.createDataFrame(valuesA,['name','id'])

valuesB = [('Lizard',1),('Dog',2),('Elephant',3),('Rat',4)]
TableB = spark.createDataFrame(valuesB,['name','id'])

TableA.show()
TableB.show()

+--------+---+
|    name| id|
+--------+---+
|     Dog|  1|
|  Monkey|  2|
|Elephant|  3|
| Penguin|  4|
+--------+---+

+--------+---+
|    name| id|
+--------+---+
|  Lizard|  1|
|     Dog|  2|
|Elephant|  3|
|     Rat|  4|
+--------+---+



In [20]:
# Create table aliases

T1 = TableA.alias('T1')
T2 = TableB.alias('T2')

In [21]:
# Inner Join on name - returns matching records from the 2 tables respectively

T1.join(T2, T1.name == T2.name, 'inner').show()


+--------+---+--------+---+
|    name| id|    name| id|
+--------+---+--------+---+
|     Dog|  1|     Dog|  2|
|Elephant|  3|Elephant|  3|
+--------+---+--------+---+



In [22]:
# Left Join Example - If you want to select all records from table 1 and return data from table 2 when it matches, you choose ‘left’

T1.join(T2, T1.name == T2.name, 'left').show()

+--------+---+--------+----+
|    name| id|    name|  id|
+--------+---+--------+----+
|  Monkey|  2|    NULL|NULL|
|     Dog|  1|     Dog|   2|
|Elephant|  3|Elephant|   3|
| Penguin|  4|    NULL|NULL|
+--------+---+--------+----+



In [23]:
# Left Join - filtering out nulls

left_join = T1.join(T2, T1.name == T2.name, 'left')
left_join.filter((T2.name).isNotNull()).show()

+--------+---+--------+---+
|    name| id|    name| id|
+--------+---+--------+---+
|     Dog|  1|     Dog|  2|
|Elephant|  3|Elephant|  3|
+--------+---+--------+---+



In [24]:
# Right Join Example - If you want to select all records from table 2 and return data from table 1 when it matches, you choose ‘right’

T1.join(T2, T1.name == T2.name, 'right').show()

+--------+----+--------+---+
|    name|  id|    name| id|
+--------+----+--------+---+
|    NULL|NULL|  Lizard|  1|
|     Dog|   1|     Dog|  2|
|Elephant|   3|Elephant|  3|
|    NULL|NULL|     Rat|  4|
+--------+----+--------+---+



In [25]:
# Full Outer Join - This shows all records from the left table and all the records from the right table and nulls where the two do not match

T1.join(T2, T1.name == T2.name,how='full').show()

+--------+----+--------+----+
|    name|  id|    name|  id|
+--------+----+--------+----+
|     Dog|   1|     Dog|   2|
|Elephant|   3|Elephant|   3|
|    NULL|NULL|  Lizard|   1|
|  Monkey|   2|    NULL|NULL|
| Penguin|   4|    NULL|NULL|
|    NULL|NULL|     Rat|   4|
+--------+----+--------+----+

